In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/PROG74040-AI-Text-Detection"
)

DATA_DIR = PROJECT_DIR / "data" / "splits"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.parquet"
VAL_FILE = DATA_DIR / "validation.parquet"
TEST_FILE = DATA_DIR / "test.parquet"

print("Train exists:", TRAIN_FILE.exists())
print("Validation exists:", VAL_FILE.exists())
print("Test exists:", TEST_FILE.exists())

Train exists: True
Validation exists: True
Test exists: True


In [4]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn

In [5]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [6]:
train_df = pd.read_parquet(TRAIN_FILE)
val_df = pd.read_parquet(VAL_FILE)
test_df = pd.read_parquet(TEST_FILE)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (341052, 3)
Validation: (73083, 3)
Test: (73083, 3)


In [7]:
train_df = train_df[["text", "generated"]].copy()
val_df = val_df[["text", "generated"]].copy()
test_df = test_df[["text", "generated"]].copy()

train_df = train_df.rename(columns={"generated": "label"})
val_df = val_df.rename(columns={"generated": "label"})
test_df = test_df.rename(columns={"generated": "label"})

train_df.head()

,text,label
0,The Importance of the Electoral College in Pre...,1
1,Voting is one of the hardest choices a person ...,0
2,Many kids believe that they should not have to...,0
3,The author supports his or her idea that study...,0
4,Automobiles have been all anyone talks about s...,0


In [8]:
deberta_train_df, _ = train_test_split(
    train_df,
    train_size=100000,
    stratify=train_df["label"],
    random_state=42
)

deberta_val_df, _ = train_test_split(
    val_df,
    train_size=20000,
    stratify=val_df["label"],
    random_state=42
)

print("DeBERTa train:", deberta_train_df.shape)
print("DeBERTa validation:", deberta_val_df.shape)

print("\nTrain distribution:")
print(deberta_train_df["label"].value_counts(normalize=True))

print("\nValidation distribution:")
print(deberta_val_df["label"].value_counts(normalize=True))

DeBERTa train: (100000, 2)
DeBERTa validation: (20000, 2)

Train distribution:
label
0    0.62764
1    0.37236
Name: proportion, dtype: float64

Validation distribution:
label
0    0.62765
1    0.37235
Name: proportion, dtype: float64


In [9]:
deberta_train_ds = Dataset.from_pandas(
    deberta_train_df,
    preserve_index=False
)

deberta_val_ds = Dataset.from_pandas(
    deberta_val_df,
    preserve_index=False
)

test_ds = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

print(deberta_train_ds)
print(deberta_val_ds)
print(test_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 100000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 20000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 73083
})


In [10]:
MODEL_NAME = "microsoft/deberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Tokenizer loaded.


In [11]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [12]:
tokenized_train = deberta_train_ds.map(
    tokenize_function,
    batched=True
)

tokenized_val = deberta_val_ds.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_ds.map(
    tokenize_function,
    batched=True
)

print(tokenized_train)
print(tokenized_val)
print(tokenized_test)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/73083 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 20000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 73083
})


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  559MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if np.isnan(logits).any():
        raise ValueError(
            "NaN logits detected. DeBERTa run is unstable."
        )

    if np.isinf(logits).any():
        raise ValueError(
            "Infinite logits detected. DeBERTa run is unstable."
        )

    predictions = np.argmax(logits, axis=-1)

    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=-1
    )[:, 1].numpy()

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            labels,
            probabilities
        )
    }

In [15]:
CHECKPOINT_DIR = MODEL_DIR / "deberta_checkpoints_100k"

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=250,

    report_to="none",

    fp16=False,

    save_total_limit=1
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  559MB            

model.safetensors: downloading bytes:           |  0.00B            

In [17]:
print("Model:", MODEL_NAME)
print("Training samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_val))
print("Test samples:", len(tokenized_test))
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("FP16:", training_args.fp16)
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint directory:", CHECKPOINT_DIR)

Model: microsoft/deberta-base
Training samples: 100000
Validation samples: 20000
Test samples: 73083
Epochs: 1
Batch size: 16
FP16: False
GPU: Tesla T4
Checkpoint directory: /content/drive/MyDrive/PROG74040-AI-Text-Detection/models/deberta_checkpoints_100k


In [18]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.007382,0.018083,0.995950,0.990936,0.998254,0.994582,0.999898


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
val_results = trainer.evaluate(
    tokenized_val
)

val_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.007382,0.018083,1,0.995950,0.990936,0.998254,0.994582,0.999898


{'eval_loss': 0.018082765862345695,
 'eval_accuracy': 0.99595,
 'eval_precision': 0.9909357504665423,
 'eval_recall': 0.9982543306029273,
 'eval_f1': 0.9945815773630343,
 'eval_roc_auc': 0.99989761686266}

In [20]:
test_results = trainer.evaluate(
    tokenized_test,
    metric_key_prefix="test"
)

test_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.007382,0.017457,1,0.996087,0.991243,0.998310,0.994764,0.999901


{'test_loss': 0.01745709590613842,
 'test_accuracy': 0.9960866412161515,
 'test_precision': 0.9912431130733025,
 'test_recall': 0.9983096314261566,
 'test_f1': 0.9947638227755401,
 'test_roc_auc': 0.9999013811470141}

In [21]:
DEBERTA_FINAL_DIR = MODEL_DIR / "deberta_final"

trainer.save_model(
    str(DEBERTA_FINAL_DIR)
)

tokenizer.save_pretrained(
    str(DEBERTA_FINAL_DIR)
)

print("DeBERTa saved to:")
print(DEBERTA_FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa saved to:
/content/drive/MyDrive/PROG74040-AI-Text-Detection/models/deberta_final


In [1]:
deberta_results_df = pd.DataFrame([{
    "model": "DeBERTa-base",
    "training_samples": 100000,
    "validation_samples": 20000,
    "accuracy": test_results["test_accuracy"],
    "precision": test_results["test_precision"],
    "recall": test_results["test_recall"],
    "f1_score": test_results["test_f1"],
    "roc_auc": test_results["test_roc_auc"]
}])

deberta_results_path = OUTPUT_DIR / "deberta_results.csv"

deberta_results_df.to_csv(
    deberta_results_path,
    index=False
)

deberta_results_df

NameError: name 'pd' is not defined